# Database Test

This notebook can be used to test the Postgres database used by the Accounting Service. I can run independently from the Accounting Sevice itself and only requires `docker` on the PATH.

In [ ]:
import java.nio.file.Path

fun String.runCommand(workingDir: Path? = notebook.workingDir): String {
    val parts = this.split(" ")
    val pb = ProcessBuilder(*parts.toTypedArray()).redirectErrorStream(true)
    if (workingDir != null) pb.directory(workingDir.toFile())
    val process = pb.start()
    return process.inputStream.bufferedReader().use { it.readText() }
}

In [ ]:
// Start underlying database
DISPLAY("docker-compose up -d".runCommand())
val jdbcUrl = "docker ps".runCommand().lines().first { output ->
    output.contains("accounting-service-postgres-")
}.run {
    val regex = Regex("0\\.0\\.0\\.0:(\\d+)->\\d+/tcp*.")
    regex.find(this)?.let {
        val localPort = it.groupValues[1]
        "jdbc:postgresql://localhost:$localPort/mydatabase"
    }
}
DISPLAY("JDBC: $jdbcUrl")

In [ ]:
val dataSrc = createDataSource(
    jdbcUrl = jdbcUrl,
    username = "myuser",
    password = "secret"
)

In [ ]:
%use ktor-client, database, dataframe

In [ ]:
dataSrc.connection.use {
    DataFrame.readSqlTable(it, "vet_payment")
}

In [ ]:
dataSrc.connection.use {
    DataFrame.readSqlTable(it, "vet_salary")
}

In [ ]:
// Proping the end points
http.get("http://localhost:8081/api/vetWorkInfo/date/2025-09-01").bodyAsText()

In [ ]:
http.get("http://localhost:8081/api/vetWorkInfo?vetId=101&date=2025-09-01").bodyAsText()